In [ ]:
import requests
import json
from google.colab import files
import time

In [ ]:
# --- 1. SETUP: Your specific list of Q-IDs ---

q_ids = [
    "Q1325291",     # Saxo Bank
    "Q20011294",    # Interactive Brokers
    "Q3366005",     # Nordnet
    "Q23680150",    # DEGIRO
    "Q481356",      # IG Group
    "Q3570310",     # XTB
    "Q1122965",     # Swissquote
    "Q3127437",     # Hargreaves Lansdown
    "Q2110465",     # Boursorama
    "Q1023871",     # CMC Markets
    "Q5324516",     # eToro
    "Q252004",      # Rabobank
    "Q104864987",   # BUX
    "Q7074354",     # OANDA (Oanda Corporation)
    "Q105475811",   # Trade Republic
    "Q22908307",    # Revolut
    "Q16826370",    # AvaTrade
    "Q103843110",   # Trading 212
    "Q193199",      # UBS
    "Q28721069",    # LYNX Broker (LYNX B.V.)
    "Q125371535",   # Moomoo
    "Q11332909",    # Forex.com
    "Q705417",      # DBS (DBS Bank)
    "Q17056725",    # Interactive Investor
    "Q18712468",    # Nutmeg (company)
    "Q65065185",    # Freetrade
    "Q7166448",     # Pepperstone
    "Q15176605",    # Plus500
    "Q1956014",     # Belfius
    "Q449233"       # PostFinance
]

In [ ]:
# --- 2. CONFIGURATION ---
BATCH_SIZE = 50
API_URL = "https://www.wikidata.org/w/api.php"

# *** FIX: We must identify our script to Wikidata or they will block it ***
HEADERS = {
    "User-Agent": "MyFinancialDataScript/1.0 (contact@example.com)"
}

def fetch_wikidata_entities(id_list):
    """
    Fetches entity data from Wikidata in batches of 50.
    Fetches ALL languages.
    """
    all_entities = {}
    total_ids = len(id_list)

    print(f"Starting fetch for {total_ids} entities (All Languages)...")

    # Process in chunks of 50
    for i in range(0, total_ids, BATCH_SIZE):
        batch = id_list[i:i + BATCH_SIZE]
        ids_string = "|".join(batch)

        params = {
            "action": "wbgetentities",
            "ids": ids_string,
            "format": "json"
        }

        try:
            # We pass 'headers=HEADERS' here to fix the 403 error
            response = requests.get(API_URL, params=params, headers=HEADERS)
            response.raise_for_status()
            data = response.json()

            if "entities" in data:
                all_entities.update(data["entities"])
                print(f"  - Fetched batch {i // BATCH_SIZE + 1} ({len(batch)} IDs)")

            time.sleep(1)

        except Exception as e:
            print(f"Error fetching batch starting with {batch[0]}: {e}")

    return all_entities

# --- 3. EXECUTION ---
entity_data = fetch_wikidata_entities(q_ids)

# --- 4. EXPORT TO JSON ---
output_filename = "financial_companies_all_languages.json"

if entity_data:
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(entity_data, f, indent=4, ensure_ascii=False)

    print(f"\nSuccess! Extracted information for {len(entity_data)} entities.")
    print(f"File saved as: {output_filename}")
    files.download(output_filename)
else:
    print("\nNo data was fetched. Please check the logs above.")

Starting fetch for 30 entities (All Languages)...
  - Fetched batch 1 (30 IDs)

Success! Extracted information for 30 entities.
File saved as: financial_companies_all_languages.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import json
import re
import requests
import time
from google.colab import files

# --- 1. PREPARATION ---
with open('financial_companies_all_languages.json', 'r') as f:
    full_data = json.load(f)

print(f"Analyzing {len(full_data)} entities...")

# --- 2. DISCOVERY: Find all P-IDs (for headers) and Q-IDs (for values) ---
all_p_ids = set()
all_referenced_q_ids = set()

for qid in full_data:
    claims = full_data[qid].get("claims", {})
    for p_id, claim_list in claims.items():
        all_p_ids.add(p_id)
        for claim in claim_list:
            try:
                val = claim.get('mainsnak', {}).get('datavalue', {}).get('value', {})
                if isinstance(val, dict) and 'id' in val and str(val['id']).startswith('Q'):
                    all_referenced_q_ids.add(val['id'])
            except: continue

# We also want to translate the main Q-IDs of the companies themselves
all_referenced_q_ids.update(full_data.keys())
all_ids_to_translate = list(all_p_ids | all_referenced_q_ids)

print(f"Found {len(all_p_ids)} unique properties and {len(all_referenced_q_ids)} unique values to translate.")

# --- 3. TRANSLATION: Fetch labels from Wikidata ---
def fetch_all_labels(id_list):
    translation_map = {}
    for i in range(0, len(id_list), 50):
        batch = id_list[i:i+50]
        params = {
            "action": "wbgetentities",
            "ids": "|".join(batch),
            "props": "labels",
            "languages": "en",
            "format": "json"
        }
        try:
            res = requests.get(API_URL, params=params, headers=HEADERS).json()
            entities = res.get("entities", {})
            for target_id, d in entities.items():
                label = d.get("labels", {}).get("en", {}).get("value", target_id)
                translation_map[target_id] = label
        except Exception as e:
            print(f"Error fetching batch: {e}")
        time.sleep(0.5)
    return translation_map

label_map = fetch_all_labels(all_ids_to_translate)

# --- 4. EXTRACTION: Build the readable table ---
def get_readable_value(claim_list):
    parts = []
    for claim in claim_list:
        try:
            mainsnak = claim.get('mainsnak', {})
            dv = mainsnak.get('datavalue', {})
            val = dv.get('value')
            v_type = dv.get('type')

            if v_type == "wikibase-entityid":
                qid = val.get('id')
                parts.append(f"{label_map.get(qid, qid)} ({qid})")
            elif v_type == "time":
                parts.append(val.get('time'))
            elif v_type == "quantity":
                parts.append(val.get('amount'))
            elif isinstance(val, str):
                parts.append(val)
            elif isinstance(val, dict) and 'text' in val:
                parts.append(val['text'])
        except: continue
    return " | ".join(list(dict.fromkeys(parts)))

rows = []
for qid, entity in full_data.items():
    # Base metadata
    row = {
        "qid": qid,
        "label_en": entity.get("labels", {}).get("en", {}).get("value", ""),
        "description_en": entity.get("descriptions", {}).get("en", {}).get("value", ""),
        "all_labels": "; ".join([f"{l}:{v['value']}" for l, v in entity.get("labels", {}).items()]),
        "wikipedia_links_count": len(entity.get("sitelinks", {}))
    }

    # Dynamic properties
    claims = entity.get("claims", {})
    for p_id in all_p_ids:
        # Create a header like "country (P17)"
        header = f"{label_map.get(p_id, p_id)} ({p_id})"
        if p_id in claims:
            row[header] = get_readable_value(claims[p_id])
        else:
            row[header] = None
    rows.append(row)

# --- 5. EXPORT ---
df_final = pd.DataFrame(rows)
output_name = "financial_entities_comprehensive_readable.csv"
df_final.to_csv(output_name, index=False, encoding="utf-8-sig")

print(f"\nSuccess! Created {output_name} with {len(df_final.columns)} columns.")
files.download(output_name)
df_final.head()

Analyzing 30 entities...
Found 172 unique properties and 229 unique values to translate.

Success! Created financial_entities_comprehensive_readable.csv with 177 columns.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,qid,label_en,description_en,all_labels,wikipedia_links_count,organizational divisions (P199),director / manager (P1037),NL CR AUT ID (P691),Google Play Store app ID (P3418),platform (P400),...,National Library Board Singapore ID (P13822),SoundCloud ID (P3040),Quora topic ID (P3417),country (P17),IPv6 routing prefix (P3793),Reddit topic ID (P11137),Corporate Number (Japan) (P3225),motto text (P1451),carbon footprint (P5991),AlternativeTo software ID (P9618)
0,Q1325291,Saxo Bank A/S,Systematic internaliser for bonds,de:Saxo Bank; cs:Saxo Bank; da:Saxo Bank; en:S...,25,None,None,None,None,None,...,None,None,None,Denmark (Q35),None,None,8010401082810,None,None,None
1,Q20011294,Interactive Brokers,American multinational brokerage firm,en:Interactive Brokers; zh-hant:盈透證券; fr:Inter...,19,None,None,None,None,None,...,None,None,None,United States (Q30),None,None,4010001100892,None,None,None
2,Q3366005,Nordnet,Nordic financial services company,fi:Nordnet; sv:Nordnet; da:Nordnet; nb:Nordnet...,7,None,None,None,None,None,...,None,None,None,Sweden (Q34),None,None,None,None,+0 | +58 | +645 | +236 | +66 | +85 | +189 | +6...,None
3,Q23680150,DEGIRO,European brokerage company,en:DEGIRO; nl:DEGIRO; fr:Degiro; de:DEGIRO; fa...,5,None,None,None,None,None,...,None,None,DEGIRO,Netherlands (Q55),None,None,None,None,None,None
4,Q481356,IG Group,Online trading company,zh-hans:IG Group; zh-hant:IG Group; zh-hk:IG G...,13,None,None,None,None,None,...,None,None,IG-Group,United Kingdom (Q145),None,None,9010401051715,None,None,None


In [ ]:
import pandas as pd

# 1. Load the readable CSV
df = pd.read_csv('financial_entities_comprehensive_readable.csv')

# 2. Create a unique identifier for columns
# We combine Name + QID to ensure that if two companies have the same name (rare), they don't overwrite each other.
df['unique_name'] = df['label_en'] + " (" + df['qid'] + ")"

# 3. Transpose
# We set the new unique name as the index, drop the old metadata columns, and flip the table.
df.set_index('unique_name', inplace=True)
df_transposed = df.drop(columns=['qid', 'label_en']).transpose()

# 4. Cleanup
# Reset index so the "Property Names" become the first column instead of the index
df_transposed.reset_index(inplace=True)
df_transposed.rename(columns={'index': 'Property'}, inplace=True)

# 5. Export
output_file = 'financial_entities_transposed.csv'
df_transposed.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Success! Transposed data saved as: {output_file}")
files.download(output_file)

# Preview
df_transposed.head()

Success! Transposed data saved as: financial_entities_transposed.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

unique_name,Property,Saxo Bank A/S (Q1325291),Interactive Brokers (Q20011294),Nordnet (Q3366005),DEGIRO (Q23680150),IG Group (Q481356),XTB (Q3570310),Swissquote (Q1122965),Hargreaves Lansdown (Q3127437),Boursorama (Q2110465),...,Moomoo (Q125371535),Forex.com (Q11332909),DBS Bank (Q705417),Interactive Investor (Q17056725),Nutmeg (Q18712468),Freetrade (Q65065185),Pepperstone (Q7166448),Plus500 (Q15176605),Belfius (Q1956014),PostFinance (Q449233)
0,description_en,Systematic internaliser for bonds,American multinational brokerage firm,Nordic financial services company,European brokerage company,Online trading company,"online trading platform, X-Trade Brokers",public company,British financial service company,French bank and stock broker,...,retail stock trading app,FX and CFD broker,multinational banking and financial services c...,British trading platform for retail investors,London based fund manager,UK based financial technology company,Australian online broker,British international financial firm,Belgian banking chain,financial institution and subsidiary of Swiss ...
1,all_labels,de:Saxo Bank; cs:Saxo Bank; da:Saxo Bank; en:S...,en:Interactive Brokers; zh-hant:盈透證券; fr:Inter...,fi:Nordnet; sv:Nordnet; da:Nordnet; nb:Nordnet...,en:DEGIRO; nl:DEGIRO; fr:Degiro; de:DEGIRO; fa...,zh-hans:IG Group; zh-hant:IG Group; zh-hk:IG G...,fr:XTB; en:XTB; pl:XTB; cs:XTB; sk:XTB; de:XTB...,de:Swissquote; gl:Swissquote; fr:Swissquote; e...,fr:Hargreaves Lansdown; en:Hargreaves Lansdown...,fr:Boursorama; es:Boursorama; ca:Boursorama; e...,...,en:Moomoo; ja:Moomoo,ja:フォレックス・ドットコム; en:Forex.com,pt:DBS Bank Limited; zh-hans:星展银行; zh-hant:星展銀...,en:Interactive Investor; fr:Interactive Investor,en:Nutmeg; fr:Nutmeg; it:Nutmeg; de:Nutmeg; es...,en:Freetrade,en:Pepperstone; fr:Pepperstone; lv:Pepperstone...,en:Plus500; de:Plus500; ar:بلوس500; es:Plus500...,nl:Belfius Bank; en:Belfius; fr:Belfius; de:Be...,cy:PostFinance; de:PostFinance; fr:PostFinance...
2,wikipedia_links_count,25,19,7,5,13,6,13,7,12,...,2,2,24,1,1,1,4,10,9,12
3,organizational divisions (P199),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,director / manager (P1037),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,director (Q1162163),NaN,NaN,NaN,NaN,NaN,Marc Raisière (Q16662863),NaN
